# DeepPoly: Certifying Neural Network Robustness on MNIST

This notebook trains a small fully-connected network on MNIST, attacks it with FGSM, then formally certifies per-image robustness using the DeepPoly abstract domain.

> Singh et al., *An Abstract Domain for Certifying Neural Networks*, POPL 2019.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import struct, gzip, urllib.request, os, time
from pathlib import Path

plt.rcParams.update({
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 120,
})

## 1. Data

In [ ]:
BASE  = "https://ossci-datasets.s3.amazonaws.com/mnist"
FILES = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
         "t10k-images-idx3-ubyte.gz",  "t10k-labels-idx1-ubyte.gz"]

os.makedirs("mnist", exist_ok=True)
for fname in FILES:
    out = Path("mnist") / fname[:-3]
    if not out.exists():
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(f"{BASE}/{fname}", str(out) + ".gz")
        with gzip.open(str(out) + ".gz", "rb") as fi, open(out, "wb") as fo:
            fo.write(fi.read())
        os.remove(str(out) + ".gz")

def read_images(path):
    with open(path, "rb") as f:
        _, n, r, c = struct.unpack(">IIII", f.read(16))
        return np.frombuffer(f.read(), np.uint8).reshape(n, r*c).astype(np.float32) / 255.0

def read_labels(path):
    with open(path, "rb") as f:
        _, n = struct.unpack(">II", f.read(8))
        return np.frombuffer(f.read(), np.uint8).astype(np.int32)

X_train = read_images("mnist/train-images-idx3-ubyte")
y_train = read_labels("mnist/train-labels-idx1-ubyte")
X_test  = read_images("mnist/t10k-images-idx3-ubyte")
y_test  = read_labels("mnist/t10k-labels-idx1-ubyte")
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 2. Network

Architecture: **784 → 128 → 64 → 10**. Hidden layers use ReLU; the output layer uses sigmoid so activations live in $(0,1)$, compatible with MSE against one-hot targets. Weights are initialised uniformly in $[-0.1, 0.1]$ to avoid first-layer saturation.

In [ ]:
def sigmoid(x):   return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
def relu(x):      return np.maximum(0.0, x)

class Net:
    def __init__(self, dims, scale=0.1, seed=42):
        rng = np.random.default_rng(seed)
        self.W = [rng.uniform(-scale, scale, (dims[i], dims[i+1])).astype(np.float32)
                  for i in range(len(dims)-1)]
        self.b = [np.zeros((1, dims[i+1]), np.float32) for i in range(len(dims)-1)]

    def forward(self, X):
        self.zs, self.as_ = [], [X]
        a = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            self.zs.append(z)
            a = sigmoid(z) if i == len(self.W)-1 else relu(z)
            self.as_.append(a)
        return a

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

    def backward(self, y_oh, lr):
        n = self.as_[0].shape[0]
        s = self.as_[-1]
        delta = 2.0/n * (s - y_oh) * s * (1.0 - s)
        for i in reversed(range(len(self.W))):
            dW = self.as_[i].T @ delta
            db = delta.sum(axis=0, keepdims=True)
            if i > 0:
                delta = (delta @ self.W[i].T) * (self.zs[i-1] > 0)
            else:
                self._input_grad = delta @ self.W[0].T
            self.W[i] -= lr * dW
            self.b[i]  -= lr * db

net = Net([784, 128, 64, 10])
print("Network: 784 → 128 (ReLU) → 64 (ReLU) → 10 (Sigmoid)")

## 3. Training

In [ ]:
EPOCHS, LR, BATCH = 30, 0.5, 32
rng = np.random.default_rng(0)

def onehot(y, n=10):
    oh = np.zeros((len(y), n), np.float32)
    oh[np.arange(len(y)), y] = 1.0
    return oh

history = []
for epoch in range(EPOCHS):
    t0  = time.time()
    idx = rng.permutation(len(X_train))
    losses = []
    for s in range(0, len(X_train), BATCH):
        b = idx[s:s+BATCH]
        net.forward(X_train[b])
        oh = onehot(y_train[b])
        losses.append(np.mean((net.as_[-1] - oh)**2))
        net.backward(oh, LR)
    acc = np.mean(net.predict(X_test) == y_test) * 100
    history.append((np.mean(losses), acc))
    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={np.mean(losses):.5f}  acc={acc:.2f}%  ({time.time()-t0:.1f}s)")

print(f"\nFinal test accuracy: {history[-1][1]:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ep = range(1, EPOCHS+1)

ax1.plot(ep, [h[0] for h in history], color='steelblue', lw=1.8)
ax1.set(xlabel='Epoch', ylabel='MSE Loss', title='Training Loss')

ax2.plot(ep, [h[1] for h in history], color='steelblue', lw=1.8)
ax2.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Test Accuracy')

plt.tight_layout()
plt.show()

## 4. FGSM Attack

The Fast Gradient Sign Method perturbs each pixel in the direction that maximally increases the loss:

$$x_{\text{adv}} = \operatorname{clip}\!\left(x + \varepsilon \cdot \operatorname{sign}(\nabla_{x}\, L),\; 0,\; 1\right)$$

The perturbation is bounded by $\varepsilon$ in the $L_\infty$ norm.

In [ ]:
def fgsm(net, x, y_true, eps):
    net.forward(x.reshape(1, -1))
    net.backward(onehot([y_true]), lr=0)
    return np.clip(x + eps * np.sign(net._input_grad.ravel()), 0.0, 1.0)

N, EPS_ATK = 100, 0.10
results = []
for i in range(N):
    x, y = X_test[i], int(y_test[i])
    pc   = int(net.predict(x.reshape(1,-1))[0])
    xadv = fgsm(net, x, y, EPS_ATK)
    pa   = int(net.predict(xadv.reshape(1,-1))[0])
    results.append(dict(idx=i, true=y, pc=pc, pa=pa,
                        attacked=(pc==y and pa!=y),
                        x=x, xadv=xadv, robust=False, margin=0.0))

n_correct  = sum(r['pc'] == r['true'] for r in results)
n_attacked = sum(r['attacked'] for r in results)
print(f"{N} images  |  {n_correct} correct  |  {n_attacked} successfully attacked  (ε={EPS_ATK})")

### Visualising adversarial perturbations

The difference between original and adversarial image is amplified ×5 below so the perturbation is visible.

In [ ]:
examples = [r for r in results if r['attacked']][:6]
fig, axes = plt.subplots(3, len(examples), figsize=(12, 4.5))

for col, r in enumerate(examples):
    orig  = r['x'].reshape(28, 28)
    adv   = r['xadv'].reshape(28, 28)
    delta = np.clip((adv - orig) * 5 + 0.5, 0, 1)
    for row, (img, cm, title) in enumerate([
        (orig,  'gray',    f"true={r['true']}"),
        (adv,   'gray',    f"pred={r['pa']}"),
        (delta, 'bwr',     "δ×5"),
    ]):
        ax = axes[row][col]
        ax.imshow(img, cmap=cm, vmin=0, vmax=1, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(['Original', 'Adversarial', 'Perturbation'][row],
                          fontsize=9)
        if row == 0:
            ax.set_title(title, fontsize=9)

fig.suptitle(f"FGSM perturbations  (ε={EPS_ATK})", fontsize=11)
plt.tight_layout()
plt.show()

## 5. DeepPoly Verifier

Rather than enumerating all inputs in the $L_\infty$ ball $\{x': \|x'-x\|_\infty \le \varepsilon\}$, DeepPoly propagates a **symbolic abstract element** through the network. Each neuron $i$ in layer $k$ maintains:

$$\hat{l}_i = \alpha_i^\top x + a_i \;\le\; z_i \;\le\; \hat{u}_i = \beta_i^\top x + b_i$$

along with concrete interval bounds $[l_i, u_i]$ obtained by evaluating those expressions over the input box.

**Transformers:**

| Layer | Symbolic bounds | Concrete bounds |
|---|---|---|
| Affine | Exact via backsubstitution into previous layer's expressions | Evaluate over input box |
| ReLU ($u\le0$) | $\hat{l}=\hat{u}=0$ | $l=u=0$ |
| ReLU ($l\ge0$) | Identity (pass through) | Pass through |
| ReLU (mixed) | Upper: $\frac{u}{u-l}\hat{u}$; Lower: $0$ or identity (min-area) | $l=0$, $u$ from upper |
| Sigmoid | Concrete only (monotone: $\sigma$ applied to $[l,u]$) | $[\sigma(l), \sigma(u)]$ |

**Decision:** the network is **certified robust** at $x$ iff $\hat{l}_{k^*} > \hat{u}_j$ for every $j \ne k^*$, where $k^*$ is the true class.

In [ ]:
def eval_concrete(lbc, ubc, lbb, ubb, il, iu):
    l = lbb + np.where(lbc >= 0, lbc * il, lbc * iu).sum(1)
    u = ubb + np.where(ubc >= 0, ubc * iu, ubc * il).sum(1)
    return l, u

def dp_first_affine(W, b, il, iu):
    # W: (n_in, n_out)  →  coefs are W^T
    lbc = ubc = W.T.copy()
    lbb = ubb = b.copy()
    l, u = eval_concrete(lbc, ubc, lbb, ubb, il, iu)
    return lbc, ubc, lbb, ubb, l, u

def dp_affine(prev, W, b, il, iu):
    lbc, ubc, lbb, ubb = prev[:4]
    Wp, Wn = np.maximum(W, 0), np.minimum(W, 0)
    nlbc = Wp.T @ lbc + Wn.T @ ubc
    nubc = Wp.T @ ubc + Wn.T @ lbc
    nlbb = Wp.T @ lbb + Wn.T @ ubb + b
    nubb = Wp.T @ ubb + Wn.T @ lbb + b
    l, u = eval_concrete(nlbc, nubc, nlbb, nubb, il, iu)
    return nlbc, nubc, nlbb, nubb, l, u

def dp_relu(prev, il, iu):
    lbc, ubc, lbb, ubb, pl, pu = prev
    n, m = lbc.shape
    nlbc = np.zeros_like(lbc);  nubc = np.zeros_like(ubc)
    nlbb = np.zeros(n, np.float32); nubb = np.zeros(n, np.float32)
    nl   = np.zeros(n, np.float32); nu   = np.zeros(n, np.float32)

    for i in range(n):
        li, ui = pl[i], pu[i]
        if ui <= 0:
            pass                                          # zero
        elif li >= 0:
            nlbc[i]=lbc[i]; nubc[i]=ubc[i]              # identity
            nlbb[i]=lbb[i]; nubb[i]=ubb[i]
            nl[i]=li; nu[i]=ui
        else:
            slope = ui / (ui - li)
            nubc[i] = slope * ubc[i]                    # upper: slope*x + intercept
            nubb[i] = slope * ubb[i] + (-ui*li/(ui-li))
            nu[i]   = ui
            if ui > -li:                                 # lower: identity if smaller area
                nlbc[i]=lbc[i]; nlbb[i]=lbb[i]; nl[i]=li

    return nlbc, nubc, nlbb, nubb, nl, nu

def deeppoly_verify(net, x, true_label, eps):
    il = np.clip(x - eps, 0.0, 1.0).astype(np.float32)
    iu = np.clip(x + eps, 0.0, 1.0).astype(np.float32)

    curr = dp_first_affine(net.W[0], net.b[0].ravel(), il, iu)
    curr = dp_relu(curr, il, iu) if len(net.W) > 1 else (
        *curr[:4], sigmoid(curr[4]), sigmoid(curr[5]))

    for k in range(1, len(net.W)):
        curr = dp_affine(curr, net.W[k], net.b[k].ravel(), il, iu)
        if k < len(net.W) - 1:
            curr = dp_relu(curr, il, iu)
        else:
            curr = (*curr[:4], sigmoid(curr[4]), sigmoid(curr[5]))

    l_out, u_out = curr[4], curr[5]
    margin = min(l_out[true_label] - u_out[j] for j in range(10) if j != true_label)
    return margin > 0, float(margin)

print("DeepPoly verifier ready.")

## 6. Certification

In [ ]:
EPS_VERIFY = 0.05

for r in results:
    r['robust'], r['margin'] = deeppoly_verify(net, r['x'], r['true'], EPS_VERIFY)

n_robust = sum(r['robust'] for r in results)
print(f"{N} images  |  {n_correct} correct  |  "
      f"{n_attacked} attacked (ε={EPS_ATK})  |  "
      f"{n_robust} certified robust (ε={EPS_VERIFY})")

## 7. Results

In [ ]:
# ── Results grid ──────────────────────────────────────────────────────────────
COLS, ROWS = 10, 10
fig, axes = plt.subplots(ROWS, COLS, figsize=(14, 14))

COLORS = {'robust': '#2e7d32', 'attacked': '#b71c1c',
          'unverif': '#e65100', 'wrong': '#546e7a'}

for i, r in enumerate(results[:ROWS*COLS]):
    ax = axes[i//COLS][i%COLS]
    if r['attacked']:
        img, c, lbl = r['xadv'].reshape(28,28), COLORS['attacked'], f"{r['true']}→{r['pa']}"
    elif r['pc'] != r['true']:
        img, c, lbl = r['x'].reshape(28,28),    COLORS['wrong'],    f"{r['true']}→{r['pc']}"
    elif r['robust']:
        img, c, lbl = r['x'].reshape(28,28),    COLORS['robust'],   str(r['true'])
    else:
        img, c, lbl = r['x'].reshape(28,28),    COLORS['unverif'],  str(r['true'])

    ax.imshow(img, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    for sp in ax.spines.values():
        sp.set_edgecolor(c); sp.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(lbl, fontsize=7, color=c, pad=2)

patches = [
    mpatches.Patch(color=COLORS['robust'],   label=f'Certified robust ({n_robust})'),
    mpatches.Patch(color=COLORS['attacked'], label=f'Attacked ({n_attacked})'),
    mpatches.Patch(color=COLORS['unverif'],  label='Unverified'),
    mpatches.Patch(color=COLORS['wrong'],    label='Misclassified'),
]
fig.legend(handles=patches, loc='lower center', ncol=4,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))
fig.suptitle(
    f"DeepPoly certification on MNIST   "
    f"(attack ε={EPS_ATK}, verify ε={EPS_VERIFY})",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary charts ────────────────────────────────────────────────────────────
n_wrong   = sum(r['pc'] != r['true'] for r in results)
n_unverif = N - n_robust - n_attacked - n_wrong

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

labels = ['Certified\nRobust', 'Attacked', 'Unverified', 'Misclassified']
sizes  = [n_robust, n_attacked, n_unverif, n_wrong]
colors = [COLORS['robust'], COLORS['attacked'], COLORS['unverif'], COLORS['wrong']]
wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels, colors=colors,
    autopct=lambda p: f'{p:.0f}%\n({int(round(p*N/100))})',
    startangle=90, textprops={'fontsize': 9}
)
ax1.set_title(f'Outcome breakdown  (N={N})', fontsize=10)

margins_r = [r['margin'] for r in results if r['robust']]
margins_u = [r['margin'] for r in results
             if not r['robust'] and r['pc']==r['true'] and not r['attacked']]
if margins_r:
    ax2.hist(margins_r, bins=12, color=COLORS['robust'],
             alpha=0.75, label='Certified', edgecolor='white', lw=0.5)
if margins_u:
    ax2.hist(margins_u, bins=12, color=COLORS['unverif'],
             alpha=0.65, label='Unverified', edgecolor='white', lw=0.5)
ax2.axvline(0, color='black', lw=1, ls='--', label='Decision boundary')
ax2.set(xlabel='Certification margin', ylabel='Count',
        title='Margin distribution  (> 0 → certified)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Summary  (N={N})")
print(f"  Correct          : {n_correct}")
print(f"  Attacked  ε={EPS_ATK}  : {n_attacked}")
print(f"  Certified ε={EPS_VERIFY}  : {n_robust}")
if margins_r:
    print(f"  Mean margin      : {np.mean(margins_r):.4f}")